<a href="https://colab.research.google.com/github/taguch1s/my_kaggle_docker/blob/master/notebook/jigsaw_deberta_v3_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!uv pip install kaggle
!mkdir /root/.kaggle
!cp /content/drive/MyDrive/kaggle/kaggle.json /root/.kaggle
!chmod 600 /root/.kaggle/kaggle.json


Using Python 3.12.12 environment at: /usr
Audited 1 package in 455ms


In [ ]:
!kaggle competitions download -c jigsaw-agile-community-rules

  0% 0.00/695k [00:00<?, ?B/s]
100% 695k/695k [00:00<00:00, 1.24GB/s]


In [ ]:
!unzip -o /content/jigsaw-agile-community-rules.zip -d input/

Archive:  /content/jigsaw-agile-community-rules.zip
  inflating: input/sample_submission.csv  
  inflating: input/test.csv          
  inflating: input/train.csv         


In [ ]:
!uv pip install wandb -qU

In [ ]:
import wandb
from google.colab import userdata
wandb_api_key = userdata.get('WANDB_API_KEY')
!wandb login $wandb_api_key

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

## utils

In [ ]:
%%writefile utils.py
import pandas as pd
import re

def url_to_semantics(text: str) -> str:
    if not isinstance(text, str):
        return ""

    url_pattern = r'https?://[^\s/$.?#].[^\s]*'
    urls = re.findall(url_pattern, text)

    if not urls:
        return ""

    all_semantics = []
    seen_semantics = set()

    for url in urls:
        url_lower = url.lower()

        domain_match = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", url_lower)
        if domain_match:
            full_domain = domain_match.group(1)
            parts = full_domain.split('.')
            for part in parts:
                if part and part not in seen_semantics and len(part) > 3: # Avoid short parts like 'www'
                    all_semantics.append(f"domain:{part}")
                    seen_semantics.add(part)

        # 2. Extract path parts
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", url_lower)
        path_parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()] # Split by common delimiters

        for part in path_parts:
            # Clean up potential file extensions or query params
            part_clean = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", part)
            if part_clean and part_clean not in seen_semantics and len(part_clean) > 3:
                all_semantics.append(f"path:{part_clean}")
                seen_semantics.add(part_clean)

    if not all_semantics:
        return ""

    return f"\nURL Keywords: {' '.join(all_semantics)}"


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []

    flatten.append(train_dataset[["body", "rule", "subreddit","rule_violation"]].copy())

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"

            if col_name in train_dataset.columns:
                sub_dataset = train_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0

                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]

                if not sub_dataset.empty:
                    flatten.append(sub_dataset)

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"

            if col_name in test_dataset.columns:
                sub_dataset = test_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0

                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]

                if not sub_dataset.empty:
                    flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(subset=['body', 'rule', 'subreddit'], ignore_index=True)
    dataframe.drop_duplicates(subset=['body','rule'],keep='first',inplace=True)

    return dataframe.sample(frac=1, random_state=42).reset_index(drop=True)

Writing utils.py


## custom model

In [ ]:
%%writefile models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel
from transformers.modeling_outputs import SequenceClassifierOutput

class JigsawModelTextW(nn.Module):
    def __init__(self, model_name: str, num_labels: int, freeze_layers: bool = True):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "num_labels": num_labels})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.regressor = nn.Linear(self.config.hidden_size, num_labels)

        # Cross attention: Body attends to Rule
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=self.config.hidden_size,
            num_heads=8,
            batch_first=True
        )

        # Feed-Forward層（Transformerスタイル）
        self.cross_attn_norm = nn.LayerNorm(self.config.hidden_size)
        self.cross_attn_dropout = nn.Dropout(0.1)
        # 🔥 4つの特徴量を結合するためのFeed-Forward Network
        self.feed_forward = nn.Sequential(
            nn.Linear(self.config.hidden_size * 2, self.config.hidden_size * 4),  # 拡張
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.hidden_size * 4, self.config.hidden_size),
        )
        self.ff_norm = nn.LayerNorm(self.config.hidden_size)
        self.ff_dropout = nn.Dropout(0.1)

        self.loss_fn = nn.CrossEntropyLoss()

        if freeze_layers:
            self._freeze_top_half_layers()

    def _freeze_top_half_layers(self):
        """先頭半分の層をfreeze"""
        n_layers = self.config.num_hidden_layers
        freeze_count = n_layers // 2 + 1

        print(f"Freezing top {freeze_count} layers out of {n_layers} total layers")

        for i in range(freeze_count):
            for name, param in self.backbone.encoder.layer[i].named_parameters():
                param.requires_grad = False

    def _extract_text_tokens(self, token_type_ids, attention_mask, last_hidden_state):
        """token_type_idsを使用してtext部分を抽出"""
        batch_size = token_type_ids.size(0)
        text_embeddings = []

        for i in range(batch_size):
            # text部分（token_type_id == 1）のマスクを作成
            text_mask = (token_type_ids[i] == 1) & (attention_mask[i] == 1)

            if text_mask.any():
                # text部分の埋め込みを抽出
                text_tokens = last_hidden_state[i][text_mask]
                text_pooled = text_tokens.mean(dim=0)
            else:
                text_pooled = torch.zeros(self.config.hidden_size, device=token_type_ids.device)

            text_embeddings.append(text_pooled)

        return torch.stack(text_embeddings)


    def _pool_by_token_type(self, token_type_ids, attention_mask, last_hidden_state):
        # last_hidden_state: [B,S,H], attention_mask: [B,S], token_type_ids: [B,S] or None
        use_type_ids = (
            token_type_ids is not None
            and token_type_ids.dim() == 2
            and torch.any(token_type_ids > 0)
        )
        if use_type_ids:
            text_mask_2d = (token_type_ids == 1) & (attention_mask == 1)  # [B,S] boolean AND
            rule_mask_2d = (token_type_ids == 0) & (attention_mask == 1)
        else:
            raise Exception

        # text
        text_mask = text_mask_2d.unsqueeze(-1).type_as(last_hidden_state)      # [B,S,1] -> broadcast OK
        text_num = (last_hidden_state * text_mask).sum(dim=1)                  # [B,H]
        text_feat = text_num / text_mask.sum(dim=1).clamp(min=1e-9)                        # [B,1]

        # rule
        rule_mask = rule_mask_2d.unsqueeze(-1).type_as(last_hidden_state)      # [B,S,1] -> broadcast OK
        rule_num = (last_hidden_state * rule_mask).sum(dim=1)                  # [B,H]
        rule_feat = rule_num / rule_mask.sum(dim=1).clamp(min=1e-9)

        return rule_feat, text_feat

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,  # DeBERTa/RoBERTaはNoneでOK
        labels=None
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,  # 念のため明示
            return_dict=True
        )
        total_features, _ = outputs.last_hidden_state.max(1)  # tuple: [layer0..last]

        # Text部分のみのpooling
        rule_features, text_features = self._pool_by_token_type(
            token_type_ids, attention_mask, outputs.last_hidden_state
        )  # [batch_size, hidden_size]

        # Option 1: 現在の方式（pooled features同士のattention）
        attended_text, _ = self.cross_attention(
            query=text_features.unsqueeze(1),
            key=rule_features.unsqueeze(1),
            value=text_features.unsqueeze(1),
        )
        attended_text = attended_text.squeeze(1)

        # concat
        # Transformer-style processing
        attended_text = self.cross_attn_norm(attended_text)
        attended_text = self.cross_attn_dropout(attended_text)

        ff_output = self.feed_forward(torch.cat([attended_text, total_features], dim=1))
        final_features = self.ff_norm(ff_output)
        final_features = self.ff_dropout(final_features)

        logits = self.regressor(final_features)

        loss = None
        if labels is not None:
            # 分類（0/1）のときはCrossEntropy
            loss = self.loss_fn(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)


class JigsawModelConcatrate(nn.Module):
    def __init__(self, model_name: str, num_labels: int, freeze_layers: bool = True):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "num_labels": num_labels})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.regressor = nn.Linear(self.config.hidden_size * 4, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

        if freeze_layers:
            self._freeze_top_half_layers()

    def _freeze_top_half_layers(self):
        """先頭半分の層をfreeze"""
        n_layers = self.config.num_hidden_layers
        freeze_count = n_layers // 2 + 1

        print(f"Freezing top {freeze_count} layers out of {n_layers} total layers")

        for i in range(freeze_count):
            for name, param in self.backbone.encoder.layer[i].named_parameters():
                param.requires_grad = False

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,  # DeBERTa/RoBERTaはNoneでOK
        labels=None
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,  # 念のため明示
            return_dict=True
        )
        sequence_output = torch.cat([outputs["hidden_states"][-1*i][:,0] for i in range(1, 4+1)], dim=1)  # concatenate
        logits = self.regressor(sequence_output)

        loss = None
        if labels is not None:
            # 分類（0/1）のときはCrossEntropy
            loss = self.loss_fn(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

class JigsawModelCustomHeader(nn.Module):
    def __init__(self, model_name: str, header:str, num_labels: int, freeze_layers: bool = False):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "num_labels": num_labels})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.regressor = nn.Linear(self.config.hidden_size, num_labels)
        self.header = header

        if self.header == "lstm":
            self.lstm = nn.LSTM(self.config.hidden_size, self.config.hidden_size, batch_first=True)
        elif self.header == "attention":
            # Attention layer
            self.attention = nn.MultiheadAttention(
                embed_dim=self.config.hidden_size,
                num_heads=8,  # 通常8または16
                dropout=0.1,
                batch_first=True  # 重要: batch_first=True
            )

            # Layer normalization and dropout
            self.layer_norm = nn.LayerNorm(self.config.hidden_size)
            self.dropout = nn.Dropout(0.1)

        # 共通
        self.regressor = nn.Linear(self.config.hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()


        if freeze_layers:
            self._freeze_top_half_layers()

    def _freeze_top_half_layers(self):
        """先頭半分の層をfreeze"""
        n_layers = self.config.num_hidden_layers
        freeze_count = n_layers // 2 + 1

        print(f"Freezing top {freeze_count} layers out of {n_layers} total layers")

        for i in range(freeze_count):
            for name, param in self.backbone.encoder.layer[i].named_parameters():
                param.requires_grad = False

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,  # DeBERTa/RoBERTaはNoneでOK
        labels=None
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,  # 念のため明示
            return_dict=True
        )
        if self.header == "maxpooling":
            sequence_output, _ = outputs['last_hidden_state'].max(1)  # max pooling
        elif self.header == "meanpooling":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs['last_hidden_state'].size()).float()
            sequence_output = torch.sum(outputs['last_hidden_state'] * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        elif self.header == "lstm":
            out, _ = self.lstm(outputs['last_hidden_state'], None)
            sequence_output = out[:, -1, :]
        elif self.header == "attention":
            # 最後の隠れ状態を取得
            last_hidden_state = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]

            # Self-attention適用
            # key_padding_maskでPADトークンをマスク
            key_padding_mask = ~attention_mask.bool()  # PADトークンをTrue

            attn_output, attn_weights = self.attention(
                query=last_hidden_state,
                key=last_hidden_state,
                value=last_hidden_state,
                key_padding_mask=key_padding_mask
            )

            # Residual connection + Layer Norm
            attn_output = self.layer_norm(attn_output + last_hidden_state)
            attn_output = self.dropout(attn_output)

            # [CLS]トークン（最初のトークン）を取得
            sequence_output = attn_output[:, 0, :]  # [batch_size, hidden_size]

        logits = self.regressor(sequence_output)

        loss = None
        if labels is not None:
            # 分類（0/1）のときはCrossEntropy
            loss = self.loss_fn(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

Writing models.py


## no cv

In [ ]:
%%writefile train_deberta.py
import os
import pandas as pd
import torch
import random
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from utils import get_dataframe_to_train, url_to_semantics

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CFG:
    model_name_or_path = "/kaggle/input/huggingfacedebertav3variants/deberta-v3-base"
    data_path = "/kaggle/input/jigsaw-agile-community-rules/"
    output_dir = "./deberta_v3_small_final_model"

    EPOCHS = 3
    LEARNING_RATE = 2e-5

    MAX_LENGTH = 512
    BATCH_SIZE = 8

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

def main():
    seed_everything(42)
    training_data_df = get_dataframe_to_train(CFG.data_path)
    # training_data_df, valid_df = train_test_split(full_df,test_size=0.2,stratify=full_df['rule'],random_state=42)
    print(f"Training dataset (from examples only) size: {len(training_data_df)}")

    test_df_for_prediction = pd.read_csv(f"{CFG.data_path}/test.csv")

    training_data_df['body_with_url'] = training_data_df['body'].apply(lambda x: x + url_to_semantics(x))
    training_data_df['input_text'] = training_data_df['rule'] + "[SEP]" + training_data_df['body_with_url']

    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    train_encodings = tokenizer(training_data_df['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    train_labels = training_data_df['rule_violation'].tolist()
    train_dataset = JigsawDataset(train_encodings, train_labels)

    model = AutoModelForSequenceClassification.from_pretrained(CFG.model_name_or_path, num_labels=2)

    training_args = TrainingArguments(
        output_dir=CFG.output_dir,
        num_train_epochs=CFG.EPOCHS,
        learning_rate=CFG.LEARNING_RATE,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        report_to="none",
        save_strategy="no",  #这一行加上这个 save_strategy="no"
        logging_steps=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    trainer.train()

    test_df_for_prediction['body_with_url'] = test_df_for_prediction['body'].apply(lambda x: x + url_to_semantics(x))
    test_df_for_prediction['input_text'] = test_df_for_prediction['rule'] + "[SEP]" + test_df_for_prediction['body_with_url']

    test_encodings = tokenizer(test_df_for_prediction['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    test_dataset = JigsawDataset(test_encodings)

    predictions = trainer.predict(test_dataset)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

    submission_df = pd.DataFrame({
        "row_id": test_df_for_prediction["row_id"],
        "rule_violation": probs
    })
    submission_df.to_csv("submission.csv", index=False)

if __name__ == "__main__":
    main()

## cv train

In [ ]:
import re

def build_llrd_param_groups(
    model,
    base_lr: float,
    layer_decay: float = 0.9,
    head_lr_scale: float = 2.0,
    weight_decay: float = 0.01,
    backbone_attr: str = "backbone",  # ← ここをあなたのクラスに合わせて
):
    """
    LLRD 用の param_groups を構築する。
    - backbone_attr 配下は層ごとに lr を割当
    - それ以外（= 自作ヘッダ）はヘッド扱い（lr = base_lr * head_lr_scale）
    """
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]

    # まず backbone 配下から層数を推定
    layer_indices = []
    for n, _ in model.named_parameters():
        if not n.startswith(f"{backbone_attr}."):
            continue
        # 代表的な命名をカバー（bert/roberta/deberta/mpnet/distilbert など）
        m = (
            re.search(r"\bencoder\.layer\.(\d+)\.", n) or
            re.search(r"\btransformer\.layer\.(\d+)\.", n) or
            re.search(r"\bdeberta\.encoder\.layer\.(\d+)\.", n) or
            re.search(r"\broberta\.encoder\.layer\.(\d+)\.", n) or
            re.search(r"\bbert\.encoder\.layer\.(\d+)\.", n)
        )
        if m:
            layer_indices.append(int(m.group(1)))
    num_layers = (max(layer_indices) + 1) if layer_indices else 0

    def guess_layer_id(param_name: str) -> int:
        # ヘッド判定: backbone 以外はすべてヘッド扱い
        if not param_name.startswith(f"{backbone_attr}."):
            return num_layers  # head

        # embeddings（最下層）
        if re.search(rf"^{backbone_attr}\.embeddings\.", param_name):
            return -1

        # transformer 層番号の抽出
        m = (
            re.search(r"\bencoder\.layer\.(\d+)\.", param_name) or
            re.search(r"\btransformer\.layer\.(\d+)\.", param_name) or
            re.search(r"\bdeberta\.encoder\.layer\.(\d+)\.", param_name) or
            re.search(r"\broberta\.encoder\.layer\.(\d+)\.", param_name) or
            re.search(r"\bbert\.encoder\.layer\.(\d+)\.", param_name)
        )
        if m:
            return int(m.group(1))

        # backbone 配下だが層が特定できない → 上層扱いに寄せる
        return num_layers - 1 if num_layers > 0 else 0

    param_groups = []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        lid = guess_layer_id(n)

        if lid == num_layers:  # head（backbone外 or 明示的ヘッド）
            lr_here = base_lr * head_lr_scale
        elif lid == -1:        # embeddings（最下層）
            lr_here = base_lr * (layer_decay ** (num_layers))
        else:                  # 中間層 0..L-1
            lr_here = base_lr * (layer_decay ** (num_layers - 1 - lid))

        wd_here = 0.0 if any(nd in n for nd in no_decay) else weight_decay
        param_groups.append({"params": [p], "lr": lr_here, "weight_decay": wd_here})

    return param_groups

In [ ]:
import os
import pandas as pd
import torch
import random
import gc
import shutil
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from models import *
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from torch.optim import AdamW
from transformers import get_scheduler

from utils import get_dataframe_to_train, url_to_semantics

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CFG:
    name = "deberta-v3-base"
    # model_name_or_path = "intfloat/e5-base-v2"
    # model_name_or_path = "sentence-transformers/all-mpnet-base-v2"
    model_name_or_path = "microsoft/deberta-v3-base"
    data_path = "./input/"
    output_dir = "./e5-base-v2_model"

    EPOCHS = 10
    LEARNING_RATE = 2e-05
    MAX_LENGTH = 256
    BATCH_SIZE = 32

    # Cross-validation settings
    N_SPLITS = 5  # Number of folds for cross-validation
    RANDOM_STATE = 42

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

def create_comprehensive_dashboard(cv_results, fold_predictions, filename='cv_dashboard.png'):
    """Create a comprehensive dashboard with all plots in one image"""

    # Create figure with subplots
    fig = plt.figure(figsize=(20, 16))

    # Define the grid layout
    gs = plt.GridSpec(3, 3, figure=fig)

    # Plot 1: CV Metrics across folds (top left, spans 2 columns)
    ax1 = fig.add_subplot(gs[0, :2])
    folds = list(range(1, len(cv_results) + 1))
    metrics = ['f1', 'precision', 'recall', 'auc']
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

    x = np.arange(len(folds))
    width = 0.2

    for i, metric in enumerate(metrics):
        values = [result[metric] for result in cv_results]
        ax1.bar(x + i*width, values, width, label=metric.upper(), color=colors[i], alpha=0.8)

        # Add value labels
        for j, v in enumerate(values):
            ax1.text(j + i*width, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

    ax1.set_xlabel('Fold')
    ax1.set_ylabel('Score')
    ax1.set_title('Cross-Validation Metrics Across Folds', fontsize=14, fontweight='bold')
    ax1.set_xticks(x + width*1.5)
    ax1.set_xticklabels(folds)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.1)

    # Plot 2: ROC Curves for all folds (top right)
    ax2 = fig.add_subplot(gs[0, 2])
    for i, fold_pred in enumerate(fold_predictions):
        fpr, tpr, _ = roc_curve(fold_pred['true_labels'], fold_pred['probabilities'])
        auc_score = cv_results[i]['auc']
        ax2.plot(fpr, tpr, alpha=0.7, linewidth=2, label=f'Fold {i+1} (AUC = {auc_score:.3f})')

    ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
    ax2.set_xlabel('False Positive Rate')
    ax2.set_ylabel('True Positive Rate')
    ax2.set_title('ROC Curves - All Folds', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal')

    # Plot 3: Metric distributions (middle left)
    ax3 = fig.add_subplot(gs[1, 0])
    metric_data = {metric: [result[metric] for result in cv_results] for metric in metrics}
    box_plot = ax3.boxplot([metric_data[metric] for metric in metrics],
                          labels=[m.upper() for m in metrics],
                          patch_artist=True)

    # Color the boxes
    colors_box = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
    for patch, color in zip(box_plot['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax3.set_ylabel('Score')
    ax3.set_title('Metric Distributions Across Folds', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 1)

    # Plot 4: Performance summary table (middle center)
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis('tight')
    ax4.axis('off')

    # Calculate summary statistics
    summary_data = []
    for metric in metrics:
        values = [result[metric] for result in cv_results]
        summary_data.append([
            metric.upper(),
            f'{np.mean(values):.4f}',
            f'{np.std(values):.4f}',
            f'{np.min(values):.4f}',
            f'{np.max(values):.4f}'
        ])

    table = ax4.table(cellText=summary_data,
                     colLabels=['Metric', 'Mean', 'Std', 'Min', 'Max'],
                     cellLoc='center',
                     loc='center',
                     bbox=[0, 0, 1, 1])

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    ax4.set_title('Performance Summary', fontsize=14, fontweight='bold')

    # Plot 5: Training loss curves if available (middle right)
    ax5 = fig.add_subplot(gs[1, 2])
    # This would require storing loss history during training
    ax5.text(0.5, 0.5, 'Training Loss Curves\n(Enable logging to see)',
             ha='center', va='center', transform=ax5.transAxes, fontsize=12)
    ax5.set_title('Training Progress', fontsize=14, fontweight='bold')
    ax5.axis('off')

    # Plot 6: Confusion matrix for best fold (bottom left)
    ax6 = fig.add_subplot(gs[2, 0])
    best_fold_idx = np.argmax([result['f1'] for result in cv_results])
    best_fold_pred = fold_predictions[best_fold_idx]

    cm = confusion_matrix(best_fold_pred['true_labels'], best_fold_pred['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax6,
                xticklabels=['No Violation', 'Violation'],
                yticklabels=['No Violation', 'Violation'])
    ax6.set_title(f'Confusion Matrix - Best Fold (Fold {best_fold_idx + 1})',
                 fontsize=14, fontweight='bold')
    ax6.set_xlabel('Predicted')
    ax6.set_ylabel('Actual')

    # Plot 7: Class distribution (bottom center)
    ax7 = fig.add_subplot(gs[2, 1])
    all_true_labels = np.concatenate([fp['true_labels'] for fp in fold_predictions])
    class_counts = [np.sum(all_true_labels == 0), np.sum(all_true_labels == 1)]
    colors_pie = ['#FF6B6B', '#4ECDC4']
    ax7.pie(class_counts, labels=['No Violation', 'Violation'], autopct='%1.1f%%',
            colors=colors_pie, startangle=90)
    ax7.set_title('Overall Class Distribution', fontsize=14, fontweight='bold')

    # Plot 8: Fold-wise sample sizes (bottom right)
    ax8 = fig.add_subplot(gs[2, 2])
    train_sizes = [result['train_size'] for result in cv_results]
    val_sizes = [result['val_size'] for result in cv_results]

    x = np.arange(len(folds))
    ax8.bar(x - 0.2, train_sizes, 0.4, label='Training', color='#2E86AB', alpha=0.8)
    ax8.bar(x + 0.2, val_sizes, 0.4, label='Validation', color='#A23B72', alpha=0.8)

    ax8.set_xlabel('Fold')
    ax8.set_ylabel('Number of Samples')
    ax8.set_title('Sample Sizes per Fold', fontsize=14, fontweight='bold')
    ax8.set_xticks(x)
    ax8.set_xticklabels(folds)
    ax8.legend()
    ax8.grid(True, alpha=0.3)

    # Add overall title
    plt.suptitle(f'{CFG.model_name_or_path} Cross-Validation Analysis Dashboard',
                fontsize=18, fontweight='bold', y=0.98)

    # Adjust layout and save
    plt.tight_layout()
    plt.subplots_adjust(top=0.94)
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    print(f"✓ Comprehensive dashboard saved as {filename}")

def compute_metrics(eval_pred):
    """
    평가 시 사용할 메트릭을 계산합니다.

    Args:
        eval_pred: (predictions, labels) 튜플

    Returns:
        dict: 계산된 메트릭들
    """
    predictions, labels = eval_pred

    # 예측 확률 계산 (softmax 적용)
    probabilities = torch.nn.functional.softmax(torch.from_numpy(predictions), dim=1)

    # 각 Column별 AUC 계산
    auc_scores = {}
    # TODO

    # 전체 AUC (클래스 1에 대한)
    try:
        overall_auc = roc_auc_score(labels, probabilities[:, 1])
        auc_scores['overall_auc'] = overall_auc
    except ValueError:
        auc_scores['overall_auc'] = 0.0

    return auc_scores

def evaluate_model(trainer, dataset, true_labels, rules, rule_names):
    """Comprehensive evaluation of the model"""

    predictions = trainer.predict(dataset)
    pred_probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)
    pred_labels = np.argmax(predictions.predictions, axis=1)

    # Convert to numpy arrays for indexing
    true_labels = np.array(true_labels)
    rules = np.array(rules)

    # Overall metrics
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary')
    auc_score = roc_auc_score(true_labels, pred_probs[:, 1].numpy())
    cm = confusion_matrix(true_labels, pred_labels)

    return {
        'predictions': pred_labels,
        'probabilities': pred_probs[:, 1].numpy(),
        'true_labels': true_labels,
        'confusion_matrix': cm,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc_score,
        'classification_report': classification_report(true_labels, pred_labels)
    }

def run_cross_validation(full_df, tokenizer):
    """Run k-fold cross-validation"""

    # Prepare the data
    full_df['body_with_url'] = full_df['body'].apply(lambda x: x + url_to_semantics(x))
    full_df['input_text'] = full_df['rule'] + "[SEP]" + full_df['body_with_url']

    # Initialize k-fold
    skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)

    cv_results = []
    fold_predictions = []

    print(f"Starting {CFG.N_SPLITS}-fold cross-validation...")
    print("=" * 60)

    for fold, (train_idx, val_idx) in enumerate(skf.split(full_df, full_df['rule']), 1):
        print(f"\n🎯 FOLD {fold}/{CFG.N_SPLITS}")
        print("-" * 40)

        # Split data
        train_df = full_df.iloc[train_idx].reset_index(drop=True)
        val_df = full_df.iloc[val_idx].reset_index(drop=True)

        print(f"Training samples: {len(train_df)}")
        print(f"Validation samples: {len(val_df)}")
        print(f"Class balance - Train: {train_df['rule_violation'].value_counts().to_dict()}")
        print(f"Class balance - Val: {val_df['rule_violation'].value_counts().to_dict()}")

        # Tokenize
        train_encodings = tokenizer(
            train_df['input_text'].tolist(),
            truncation=True, padding=True, max_length=CFG.MAX_LENGTH
        )
        val_encodings = tokenizer(
            val_df['input_text'].tolist(),
            truncation=True, padding=True, max_length=CFG.MAX_LENGTH
        )

        train_dataset = JigsawDataset(train_encodings, train_df['rule_violation'].tolist())
        val_dataset = JigsawDataset(val_encodings, val_df['rule_violation'].tolist())

        # Initialize model for this fold
        # model = AutoModelForSequenceClassification.from_pretrained(CFG.model_name_or_path, num_labels=2)
        model = JigsawModelCustomHeader(CFG.model_name_or_path,header="meanpooling" ,num_labels=2)

        training_args = TrainingArguments(
            output_dir=f"{CFG.output_dir}_fold{fold}",
            num_train_epochs=CFG.EPOCHS,
            learning_rate=CFG.LEARNING_RATE,
            per_device_train_batch_size=CFG.BATCH_SIZE,
            per_device_eval_batch_size=CFG.BATCH_SIZE,
            warmup_ratio=0.1,
            weight_decay=0.01,
            report_to="wandb",
            run_name=f"jigsaw_{CFG.name}_fold{fold}",
            save_strategy="epoch",
            eval_strategy="epoch",
            logging_steps=10,
            metric_for_best_model="overall_auc",  # 최고 모델 선택 기준
            greater_is_better=True,          # AUC는 높을수록 좋음
            load_best_model_at_end=True, # 훈련 끝에 최고 모델 로드
            save_total_limit=1,
            fp16=True,
        )
        es_callback = EarlyStoppingCallback(early_stopping_patience=3)

        # scheduler custom
        param_groups = build_llrd_param_groups(
            model,
            base_lr=training_args.learning_rate,
            layer_decay=0.9,          # 0.95〜0.8 の範囲で調節
            head_lr_scale=2.0,        # ヘッド2倍
            weight_decay=training_args.weight_decay,
            backbone_attr="backbone", # あなたのクラスの属性名と一致させる
        )
        optimizer = AdamW(param_groups)

        # スケジューラ（TrainingArguments に合わせる）
        steps_per_epoch = (len(train_dataset) // training_args.per_device_train_batch_size)
        num_training_steps = steps_per_epoch * int(training_args.num_train_epochs)
        num_warmup_steps = int(num_training_steps * training_args.warmup_ratio)

        lr_scheduler = get_scheduler(
            name=training_args.lr_scheduler_type if isinstance(training_args.lr_scheduler_type, str)
                else training_args.lr_scheduler_type.value,
            optimizer=optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            callbacks=[es_callback],
            optimizers=(optimizer, lr_scheduler),
        )

        # Train
        trainer.train()

        # Evaluate
        fold_results = evaluate_model(
            trainer, val_dataset,
            val_df['rule_violation'].tolist(),
            val_df['rule'].tolist(),
            val_df['rule'].unique()
        )

        # Store results
        cv_results.append({
            'fold': fold,
            'precision': fold_results['precision'],
            'recall': fold_results['recall'],
            'f1': fold_results['f1'],
            'auc': fold_results['auc'],
            'train_size': len(train_df),
            'val_size': len(val_df)
        })

        # Store predictions for this fold
        fold_predictions.append({
            'fold': fold,
            'true_labels': fold_results['true_labels'],
            'predictions': fold_results['predictions'],
            'probabilities': fold_results['probabilities'],
            'rules': val_df['rule'].tolist()
        })

        print(f"Fold {fold} Results:")
        print(f"  Precision: {fold_results['precision']:.4f}")
        print(f"  Recall:    {fold_results['recall']:.4f}")
        print(f"  F1-Score:  {fold_results['f1']:.4f}")
        print(f"  AUC-ROC:   {fold_results['auc']:.4f}")

        # teardown
        del model, trainer, es_callback
        gc.collect()
        torch.cuda.empty_cache()

        # Remove saved checkpoints for this fold to save disk space.
        # load_best_model_at_end=True ensures the best weights were already
        # loaded into memory before deletion.
        try:
            shutil.rmtree(training_args.output_dir, ignore_errors=True)
        except Exception as e:
            print(f"[WARN] Failed to cleanup {training_args.output_dir}: {e}")

    return cv_results, fold_predictions

def print_cv_summary(cv_results):
    """Print comprehensive cross-validation summary"""

    print("\n" + "="*60)
    print("CROSS-VALIDATION SUMMARY")
    print("="*60)

    metrics = ['precision', 'recall', 'f1', 'auc']

    for metric in metrics:
        values = [result[metric] for result in cv_results]
        mean_val = np.mean(values)
        std_val = np.std(values)

        print(f"\n📊 {metric.upper()}:")
        print(f"  Mean: {mean_val:.4f} ± {std_val:.4f}")
        print(f"  Range: {min(values):.4f} - {max(values):.4f}")
        print(f"  Fold values: {[f'{v:.4f}' for v in values]}")

    # Overall summary
    f1_scores = [result['f1'] for result in cv_results]
    auc_scores = [result['auc'] for result in cv_results]

    print(f"\n🎯 OVERALL PERFORMANCE:")
    print(f"  Mean F1-Score: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
    print(f"  Mean AUC-ROC:  {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")

    stability = np.std(f1_scores)
    if stability < 0.03:
        stability_text = "Excellent"
    elif stability < 0.06:
        stability_text = "Good"
    elif stability < 0.1:
        stability_text = "Moderate"
    else:
        stability_text = "Variable"
    print(f"  Model Stability: {stability_text} (std: {stability:.4f})")

def main():
    gc.collect()
    torch.cuda.empty_cache()

    seed_everything(CFG.RANDOM_STATE)

    # Load data
    full_df = get_dataframe_to_train(CFG.data_path)

    print(f"📁 DATASET OVERVIEW:")
    print(f"Total samples: {len(full_df)}")
    print(f"Unique rules: {full_df['rule'].nunique()}")
    print(f"Class distribution: {full_df['rule_violation'].value_counts().to_dict()}")
    print(f"Positive rate: {full_df['rule_violation'].mean():.3f}")

    # Initialize tokenizer
    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name_or_path)

    # Run cross-validation
    cv_results, fold_predictions = run_cross_validation(full_df, tokenizer)

    # Print summary
    print_cv_summary(cv_results)

    # Create comprehensive dashboard
    print(f"\n🎨 GENERATING COMPREHENSIVE DASHBOARD...")
    create_comprehensive_dashboard(cv_results, fold_predictions, 'cv_dashboard.png')

    # Save detailed results
    cv_df = pd.DataFrame(cv_results)
    cv_df.to_csv('cross_validation_results.csv', index=False)

    # Save all predictions
    all_predictions = []
    for fold_pred in fold_predictions:
        fold_df = pd.DataFrame({
            'fold': fold_pred['fold'],
            'true_label': fold_pred['true_labels'],
            'predicted_label': fold_pred['predictions'],
            'probability': fold_pred['probabilities'],
            'rule': fold_pred['rules']
        })
        all_predictions.append(fold_df)

    predictions_df = pd.concat(all_predictions, ignore_index=True)
    predictions_df.to_csv('cv_predictions.csv', index=False)

    print(f"\n✅ CROSS-VALIDATION COMPLETED SUCCESSFULLY!")
    print(f"📊 Dashboard saved as: cv_dashboard.png")
    print(f"📁 Results saved as: cross_validation_results.csv")
    print(f"📁 Predictions saved as: cv_predictions.csv")

if __name__ == "__main__":
    main()

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Fold 4 Results:
  Precision: 0.7919
  Recall:    0.7685
  F1-Score:  0.7800
  AUC-ROC:   0.8527

🎯 FOLD 5/5
----------------------------------------
Training samples: 1500
Validation samples: 375
Class balance - Train: {1: 781, 0: 719}
Class balance - Val: {1: 191, 0: 184}


Epoch,Training Loss,Validation Loss,Overall Auc
1,0.691600,0.695116,0.629197
2,0.615100,0.595686,0.764014
3,0.428100,0.626327,0.799368
4,0.396700,0.545584,0.819486
5,0.257700,0.596909,0.839702
6,0.206100,0.658460,0.833684
7,0.120800,0.773716,0.823811
8,0.162200,0.839534,0.825006


Fold 5 Results:
  Precision: 0.7051
  Recall:    0.8639
  F1-Score:  0.7765
  AUC-ROC:   0.8397

CROSS-VALIDATION SUMMARY

📊 PRECISION:
  Mean: 0.7641 ± 0.0568
  Range: 0.7044 - 0.8553
  Fold values: ['0.8553', '0.7639', '0.7044', '0.7919', '0.7051']

📊 RECALL:
  Mean: 0.7893 ± 0.0684
  Range: 0.6667 - 0.8639
  Fold values: ['0.6667', '0.8209', '0.8266', '0.7685', '0.8639']

📊 F1:
  Mean: 0.7716 ± 0.0148
  Range: 0.7493 - 0.7914
  Fold values: ['0.7493', '0.7914', '0.7606', '0.7800', '0.7765']

📊 AUC:
  Mean: 0.8496 ± 0.0061
  Range: 0.8397 - 0.8555
  Fold values: ['0.8555', '0.8453', '0.8549', '0.8527', '0.8397']

🎯 OVERALL PERFORMANCE:
  Mean F1-Score: 0.7716 ± 0.0148
  Mean AUC-ROC:  0.8496 ± 0.0061
  Model Stability: Excellent (std: 0.0148)

🎨 GENERATING COMPREHENSIVE DASHBOARD...


/tmp/ipython-input-624488188.py:119: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  box_plot = ax3.boxplot([metric_data[metric] for metric in metrics],


✓ Comprehensive dashboard saved as cv_dashboard.png

✅ CROSS-VALIDATION COMPLETED SUCCESSFULLY!
📊 Dashboard saved as: cv_dashboard.png
📁 Results saved as: cross_validation_results.csv
📁 Predictions saved as: cv_predictions.csv


In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
!python cv_deberta.py

2025-10-25 10:45:46.164584: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761389146.184778   17713 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761389146.190681   17713 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761389146.205592   17713 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761389146.205618   17713 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761389146.205621   17713 computation_placer.cc:177] computation placer alr